# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [5]:
col = "Num_of_Loan"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad > Mediana Standard > Mediana Good.

Esto concuerda con la hipótesis de que a a mayor cantidad de préstamos mayor cantidad de malos deudores.

In [7]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [8]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [9]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Num_of_Loan_Decile,,,,,,,,,,
0,0,1,22536,0.22536,0,12192,10344,0.000000,0.541001,0.458999
1,2,2,15712,0.15712,1200,6000,8512,0.076375,0.381874,0.541752
2,3,3,15752,0.15752,992,6352,8408,0.062976,0.403250,0.533773
3,4,4,15456,0.15456,1000,5840,8616,0.064700,0.377847,0.557453
4,5,5,7528,0.07528,4256,0,3272,0.565356,0.000000,0.434644
5,6,6,8144,0.08144,4696,0,3448,0.576621,0.000000,0.423379
6,7,7,7680,0.07680,4432,0,3248,0.577083,0.000000,0.422917
7,8,9,7192,0.07192,7192,0,0,1.000000,0.000000,0.000000


No missing values found.
No infinite values found.
No duplicate rows found.


In [12]:
df[(df[continuous_variable] >= 4) & (df[continuous_variable] <= 4) & (df['Credit_Score'] == 0)].shape

(1000, 85)

Porporción de Buenos: se observa una relación negativa entre el número de buenos y la proporción de buenos,
lo cual no es razonable.

Proporción de standard: se observa una relación negativa entre el número de buenos y la proporción de deudores standard.

Proporción de malos: la proporción de malos y el número de préstamos tienen una relación positiva, lo cual es razonable.

In [15]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y los ingresos:

In [16]:
group_map = {0: "Group_1", 
             1: "Group_1", 
             2: "Group_2", 
             3: "Group_2", 
             4: "Group_3",
             5: "Group_3", 
             6: "Group_3",
             7: "Group_3"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Num_of_Loan,,,,,,,,,,
Group_1,0,2,38248,0.38248,1200,18192,18856,0.031374,0.475633,0.492993
Group_2,3,4,31208,0.31208,1992,12192,17024,0.063830,0.390669,0.545501
Group_3,5,9,30544,0.30544,20576,0,9968,0.673651,0.000000,0.326349


No missing values found.
No infinite values found.
No duplicate rows found.


In [17]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [18]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [19]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Num_of_Loan,"100,000.00",3.53,2.45,0.00,2.00,3.00,5.00,9.00


Todos los coeficientes son significativos.

Num_of_Loan_Scaled -5.8930: según lo esperado, el coeficiente es negativo: por cada préstamo adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -3.9817: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.0944: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [20]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.814781
         Iterations: 12
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -81478.
Model:                   OrderedModel   AIC:                         1.630e+05
Method:            Maximum Likelihood   BIC:                         1.630e+05
Date:                Sun, 30 Mar 2025                                         
Time:                        08:32:53                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                         coef    std err 

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Num_of_Loan_Decile -0.7327: Por cada préstamo adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -3.6158: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.1161: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [22]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.808975
         Iterations: 12
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -80897.
Model:                   OrderedModel   AIC:                         1.618e+05
Method:            Maximum Likelihood   BIC:                         1.618e+05
Date:                Sun, 30 Mar 2025                                         
Time:                        08:35:10                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                         coef    std err 

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Num_of_Loan -1.7529: Por cada préstamo adcional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -5.0260: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 1.0753: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [24]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.844426
         Iterations: 13
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -84443.
Model:                   OrderedModel   AIC:                         1.689e+05
Method:            Maximum Likelihood   BIC:                         1.689e+05
Date:                Sun, 30 Mar 2025                                         
Time:                        09:29:27                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                          coef    std err

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Num_of_Loan` usando Regresión Ordinal

| Representación             | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Num_of_Loan_Scaled`       | -5.8930                | **Log-Likelihood**: -81,478<br>**AIC**: 162,956<br>**BIC**: 162,997                   | 🔹 Buen ajuste.<br>🔹 Captura la variación continua.<br>🔹 Coeficiente negativo indica que más préstamos se asocian a menor score. |
| `Num_of_Loan_Decile`       | -0.7327                | **Log-Likelihood**: -80,897<br>**AIC**: 161,796<br>**BIC**: 161,837                   | 🔹 Mejor ajuste global.<br>🔹 Discretización permite interpretar niveles de riesgo por grupos. |
| `Grouped_Num_of_Loan`      | -1.7529                | **Log-Likelihood**: -84,443<br>**AIC**: 168,886<br>**BIC**: 168,927                   | 🔹 Peor ajuste relativo.<br>🔹 Buena interpretabilidad.<br>🔹 Menor precisión por agrupación. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
